Nível 1: Básico (URLs, Parâmetros e Cabeçalhos)
Objetivo: Praticar a construção de requisições GET simples, envio de parâmetros e manipulação de cabeçalhos.


In [ ]:
pip install requests beautifulsoup4 lxml pandas

Exercício 1.1: Faça uma requisição GET para a API pública do JSONPlaceholder ([https://jsonplaceholder.typicode.com/posts](https://jsonplaceholder.typicode.com/posts)) ou para a API do IBGE

In [ ]:
import json
import requests
import pandas as pd

from urllib.parse import parse_qs, urlencode, urlparse

print('Versão do requests:', requests.__version__)

# Teste rápido de conectividade
try:
    teste = requests.get('https://jsonplaceholder.typicode.com/posts/1', timeout=10)
    print('Internet OK — status', teste.status_code)
except requests.RequestException as erro:
    print('SEM internet:', erro)

Versão do requests: 2.32.4
Internet OK — status 200


In [ ]:
# JSONPlaceholder: API pública de testes (não precisa de chave)
resposta = requests.get('https://jsonplaceholder.typicode.com/posts/1', timeout=10)

print('Status HTTP      :', resposta.status_code)
print('Tipo do conteúdo :', resposta.headers.get('content-type'))
print('\n--- Corpo (primeiros 200 caracteres) ---')
print(resposta.text[:200])

Status HTTP      : 200
Tipo do conteúdo : application/json; charset=utf-8

--- Corpo (primeiros 200 caracteres) ---
{
  "userId": 1,
  "id": 1,
  "title": "sunt aut facere repellat provident occaecati excepturi optio reprehenderit",
  "body": "quia et suscipit\nsuscipit recusandae consequuntur expedita et cum\nrepr


Exercício 1.2: Utilize o argumento params para filtrar os resultados (por exemplo, buscando apenas o userId igual a 2 ou limitando a quantidade de resultados).


In [ ]:
filtros = {'userId': 2, '_limit': 5}
resposta = requests.get(
    'https://jsonplaceholder.typicode.com/posts',
    params=filtros,
    timeout=10
)

print('URL montada pelo requests:')
print(resposta.url)
print('\nTotal de posts retornados:', len(resposta.json()))

for post in resposta.json():
    print('-', post['title'])

URL montada pelo requests:
https://jsonplaceholder.typicode.com/posts?userId=2&_limit=5

Total de posts retornados: 5
- et ea vero quia laudantium autem
- in quibusdam tempore odit est dolorem
- dolorum ut in voluptas mollitia et saepe quo animi
- voluptatem eligendi optio
- eveniet quod temporibus


Exercício 1.3: Na sua requisição, inclua um dicionário de cabeçalhos (headers) passando um User-Agent personalizado com o nome do seu projeto.


In [ ]:
resposta = requests.get(
    'https://httpbingo.org/headers',
    headers={'User-Agent': 'atvPraticoExplorandoDadosWeb/1.0'},
    timeout=15
)

resposta.json()

{'headers': {'Accept': ['*/*'],
  'Accept-Encoding': ['gzip, deflate, br, zstd'],
  'Host': ['httpbingo.org'],
  'User-Agent': ['atvPraticoExplorandoDadosWeb/1.0'],
  'Via': ['1.1 fly.io, 1.1 fly.io'],
  'X-Forwarded-For': ['35.237.165.221, 66.241.125.232'],
  'X-Forwarded-Port': ['443'],
  'X-Forwarded-Proto': ['https'],
  'X-Forwarded-Ssl': ['on'],
  'X-Request-Start': ['t=1790082344069051']}}

In [ ]:
resposta = requests.get('https://httpbingo.org/user-agent', timeout=15)
print('Seu User-Agent padrão:')
resposta.json()

Seu User-Agent padrão:


{'user-agent': 'python-requests/2.32.4'}

Nível 2: Intermediário (JSON, Erros e Arquivos Binários)
Objetivo: Lidar com formatos de dados reais, criar códigos mais robustos contra falhas e fazer download de mídias.


Exercício 2.1: Crie uma lista com três CEPs diferentes e faça um loop para consultá-los na API do ViaCEP ([https://viacep.com.br/ws/](https://viacep.com.br/ws/){cep}/json/). Utilize o método .json() para converter as respostas e armazene os resultados em um DataFrame do Pandas.

In [ ]:
ceps = ['01001000', '20040000', '70040000'] # Lista de CEPs

cep_data = []

for cep in ceps:
    url = f"https://viacep.com.br/ws/{cep}/json/"
    try:
        resposta = requests.get(url, timeout=10)
        resposta.raise_for_status()  # Levanta um erro para códigos de status HTTP ruins (4xx ou 5xx)
        data = resposta.json()
        if 'erro' not in data: # A API do ViaCEP retorna {'erro': true} para CEPs não encontrados
            cep_data.append(data)
        else:
            print(f"CEP {cep} não encontrado ou inválido.")
    except requests.exceptions.RequestException as e:
        print(f"Erro ao consultar o CEP {cep}: {e}")

df_ceps = pd.DataFrame(cep_data)

print("DataFrame com os dados dos CEPs:")
print(df_ceps)

CEP 70040000 não encontrado ou inválido.
DataFrame com os dados dos CEPs:
         cep    logradouro complemento unidade  bairro      localidade  uf  \
0  01001-000   Praça da Sé  lado ímpar              Sé       São Paulo  SP   
1  20040-000  Rua da Ajuda                      Centro  Rio de Janeiro  RJ   

           estado   regiao     ibge   gia ddd siafi  
0       São Paulo  Sudeste  3550308  1004  11  7107  
1  Rio de Janeiro  Sudeste  3304557        21  6001  


Exercício 2.2: Escreva uma função de download segura utilizando um bloco try/except. Dentro dela, utilize resposta.raise_for_status() para capturar erros HTTP (como 404 ou 500) e imprima uma mensagem amigável caso a requisição falhe.



In [ ]:
def download_seguro(url, timeout=10):
    """
    Realiza uma requisição GET segura e retorna o objeto de resposta.
    Em caso de erro HTTP ou de requisição, imprime uma mensagem amigável.
    """
    try:
        response = requests.get(url, timeout=timeout)
        response.raise_for_status()  # Levanta um erro para códigos de status HTTP ruins (4xx ou 5xx)
        print(f"Requisição bem-sucedida para {url}")
        return response
    except requests.exceptions.HTTPError as http_err:
        print(f"Erro HTTP ao consultar {url}: {http_err}")
    except requests.exceptions.ConnectionError as conn_err:
        print(f"Erro de conexão ao consultar {url}: {conn_err}")
    except requests.exceptions.Timeout as timeout_err:
        print(f"Timeout ao consultar {url}: {timeout_err}")
    except requests.exceptions.RequestException as req_err:
        print(f"Erro inesperado ao consultar {url}: {req_err}")
    return None

# Exemplo de uso da função (descomente para testar):
# print("\n--- Testando download_seguro com URL válida ---")
# response_example_success = download_seguro('https://jsonplaceholder.typicode.com/posts/1')
# if response_example_success:
#     print("Conteúdo do exemplo:", response_example_success.json())

# print("\n--- Testando download_seguro com URL inválida (404) ---")
# response_example_fail = download_seguro('https://jsonplaceholder.typicode.com/posts/999999999999')

# print("\n--- Testando download_seguro com URL inexistente (erro de conexão/DNS) ---")
# response_example_fail_conn = download_seguro('http://url.que.nao.existe.xyz/')


Exercício 2.3: Faça uma requisição para a URL [https://picsum.photos/400/400](https://picsum.photos/400/400) para baixar uma imagem aleatória. Salve o conteúdo bruto da resposta (acessado através de resposta.content) em um arquivo local com a extensão .jpg utilizando o modo de escrita em bytes ('wb').

In [ ]:
image_url = 'https://picsum.photos/400/400'
file_name = 'imagem_aleatoria.jpg'

print(f"Tentando baixar a imagem de: {image_url}")
response_image = download_seguro(image_url, timeout=15)

if response_image:
    try:
        with open(file_name, 'wb') as f:
            f.write(response_image.content)
        print(f"Imagem salva com sucesso em '{file_name}'")
    except IOError as e:
        print(f"Erro ao salvar a imagem '{file_name}': {e}")
else:
    print("Não foi possível baixar a imagem. Verifique a URL ou sua conexão.")

Tentando baixar a imagem de: https://picsum.photos/400/400
Timeout ao consultar https://picsum.photos/400/400: HTTPSConnectionPool(host='picsum.photos', port=443): Read timed out. (read timeout=15)
Não foi possível baixar a imagem. Verifique a URL ou sua conexão.


Nível 3: Avançado (Webscraping e Ética)
Objetivo: Extrair dados de páginas HTML onde não há APIs estruturadas disponíveis, respeitando as boas práticas.


Exercício 3.1: Antes de fazer a raspagem, faça uma requisição para o arquivo robots.txt do site que deseja acessar para verificar as permissões de acesso, demonstrando responsabilidade ética na coleta de dados.

In [ ]:
robots_url = 'https://books.toscrape.com/robots.txt'

print(f"Verificando o arquivo robots.txt de: {robots_url}")
robots_response = download_seguro(robots_url)

if robots_response:
    print("\nConteúdo do robots.txt:")
    print(robots_response.text)
else:
    print("Não foi possível obter o robots.txt.")

Verificando o arquivo robots.txt de: https://books.toscrape.com/robots.txt
Erro HTTP ao consultar https://books.toscrape.com/robots.txt: 404 Client Error: Not Found for url: https://books.toscrape.com/robots.txt
Não foi possível obter o robots.txt.


Exercício 3.2: Acesse o site voltado para estudos de raspagem [https://books.toscrape.com/](https://books.toscrape.com/). Utilize a biblioteca BeautifulSoup com o analisador 'html.parser' para localizar os elementos da página. Extraia o título e o preço dos 5 primeiros livros utilizando métodos como .find() ou .select().

In [ ]:
from bs4 import BeautifulSoup

scrape_url = 'https://books.toscrape.com/'

print(f"Acessando o site: {scrape_url}")
response_html = download_seguro(scrape_url)

if response_html:
    soup = BeautifulSoup(response_html.content, 'html.parser')

    # Encontrar todos os artigos (livros) na página
    books = soup.select('article.product_pod')

    print("\nExtraindo informações dos 5 primeiros livros:")
    for i, book in enumerate(books[:5]):  # Limitar aos 5 primeiros livros
        title = book.h3.a['title']
        price = book.select_one('.price_color').get_text(strip=True)

        print(f"Livro {i+1}:")
        print(f"  Título: {title}")
        print(f"  Preço: {price}")
else:
    print("Não foi possível acessar a página para raspagem.")

Acessando o site: https://books.toscrape.com/
Requisição bem-sucedida para https://books.toscrape.com/

Extraindo informações dos 5 primeiros livros:
Livro 1:
  Título: A Light in the Attic
  Preço: £51.77
Livro 2:
  Título: Tipping the Velvet
  Preço: £53.74
Livro 3:
  Título: Soumission
  Preço: £50.10
Livro 4:
  Título: Sharp Objects
  Preço: £47.82
Livro 5:
  Título: Sapiens: A Brief History of Humankind
  Preço: £54.23


Exercício 3.3: Salve os dados extraídos dos livros em um arquivo CSV utilizando o Pandas.

In [ ]:
from bs4 import BeautifulSoup
import pandas as pd

scrape_url = 'https://books.toscrape.com/'

print(f"Acessando o site: {scrape_url}")
response_html = download_seguro(scrape_url)

if response_html:
    soup = BeautifulSoup(response_html.content, 'html.parser')

    books_data = []
    # Encontrar todos os artigos (livros) na página
    books = soup.select('article.product_pod')

    print("\nColetando dados de todos os livros da primeira página...")
    for book in books:
        title = book.h3.a['title']
        price = book.select_one('.price_color').get_text(strip=True)
        books_data.append({'Title': title, 'Price': price})

    # Criar DataFrame com os resultados
    df_books = pd.DataFrame(books_data)

    # Salvar em CSV
    csv_file_name = 'books_data.csv'
    df_books.to_csv(csv_file_name, index=False, encoding='utf-8')

    print(f"Dados dos livros salvos com sucesso em '{csv_file_name}'")
    display(df_books.head())
else:
    print("Não foi possível acessar a página para coletar os dados dos livros.")

Acessando o site: https://books.toscrape.com/
Requisição bem-sucedida para https://books.toscrape.com/

Coletando dados de todos os livros da primeira página...
Dados dos livros salvos com sucesso em 'books_data.csv'


,Title,Price
0,A Light in the Attic,£51.77
1,Tipping the Velvet,£53.74
2,Soumission,£50.10
3,Sharp Objects,£47.82
4,Sapiens: A Brief History of Humankind,£54.23


Exercício 3.4: Escolha uma página da Wikipedia que contenha uma tabela de dados (como listas de países ou populações). Utilize a função pandas.read_html() combinada com io.StringIO() para capturar a tabela da página e transformá-la diretamente em um DataFrame, sem a necessidade de usar o BeautifulSoup.